# Visual Novel — Render video bằng Google Colab

Quy trình rút gọn: **Kết nối runtime → Chạy tất cả (Run all) → Chọn ZIP khi được hỏi**. Colab tự cài thư viện, kết nối Drive, nhận gói ZIP từ `phong_video.html`, tải engine và render.

Kết quả gồm MP4, SRT, Audio Manifest, metadata và một ZIP tổng hợp; được lưu trong `MyDrive/VisualNovelRenders`. Gói ZIP đầu vào chỉ chứa timeline/WAV và tham chiếu tài nguyên; engine tải tài nguyên gốc từ GitHub.

**Lưu ý:** Colab miễn phí có giới hạn thời gian và tài nguyên, GPU không được đảm bảo. Hãy thử một đoạn ngắn trước; bản render native có thể chưa giống pixel-perfect với preview trình duyệt.

In [ ]:
import sys, subprocess, re, zipfile, base64
from pathlib import Path

print('1/5 Cài thư viện render…')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av', 'opencv-python-headless', 'pillow', 'numpy', 'requests'], check=True)

from google.colab import drive, output
import requests
print('2/5 Kết nối Google Drive để lưu kết quả…')
drive.mount('/content/drive')

print('3/5 Chọn ZIP trực tiếp từ điện thoại/máy tính…')
UPLOAD_DIR = Path('/content')
UPLOAD_STATE = {'path': None, 'name': None}
def _receive_zip_chunk(name, index, total, chunk):
    safe_name = re.sub(r'[^A-Za-z0-9._-]+', '_', Path(name).name)
    if not safe_name.lower().endswith('.zip'):
        safe_name += '.zip'
    path = UPLOAD_DIR / safe_name
    mode = 'wb' if int(index) == 0 else 'ab'
    with open(path, mode) as f:
        f.write(base64.b64decode(chunk))
    UPLOAD_STATE.update(path=path, name=safe_name)
    return {'received': int(index) + 1, 'total': int(total)}
output.register_callback('vn_upload_zip_chunk', _receive_zip_chunk)
upload_result = output.eval_js(r'''new Promise((resolve, reject) => {
  const input = document.createElement('input');
  input.type = 'file'; input.accept = '.zip,application/zip,application/x-zip-compressed';
  input.style.display = 'none'; document.body.appendChild(input);
  input.onchange = async () => {
    try {
      if (!input.files || !input.files.length) { resolve(null); return; }
      const file = input.files[0];
      const bytes = new Uint8Array(await file.arrayBuffer());
      const chunkSize = 384 * 1024;
      const total = Math.ceil(bytes.length / chunkSize);
      for (let i = 0; i < total; i++) {
        const part = bytes.subarray(i * chunkSize, Math.min((i + 1) * chunkSize, bytes.length));
        let binary = ''; for (let j = 0; j < part.length; j++) binary += String.fromCharCode(part[j]);
        const b64 = btoa(binary);
        await google.colab.kernel.invokeFunction('vn_upload_zip_chunk', [file.name, i, total, b64], {});
      }
      resolve({name: file.name, size: file.size, chunks: total});
    } catch (e) { reject(String(e)); }
    finally { input.remove(); }
  };
  input.click();
})''')
if not upload_result or not UPLOAD_STATE['path']:
    raise RuntimeError('Chưa chọn được ZIP. Hãy chạy lại ô này và chọn tệp từ Bộ nhớ/Download trên thiết bị.')
ZIP_PATH = UPLOAD_STATE['path']
if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError('Tệp tải lên không phải ZIP hợp lệ hoặc tải chưa hoàn tất.')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f'ZIP bị lỗi tại mục: {bad}')
    if 'timeline.json' not in z.namelist():
        raise RuntimeError('ZIP không có timeline.json. Hãy chọn đúng gói xuất từ phòng tạo video.')
print('Đã nhận ZIP trực tiếp:', ZIP_PATH.name, round(ZIP_PATH.stat().st_size/1024/1024, 1), 'MB; chia thành', upload_result['chunks'], 'phần.')

print('4/5 Tải engine render từ GitHub…')
ENGINE_URL = 'https://raw.githubusercontent.com/nhothicam01-png/visual-novel-game/main/colab-render/renderer.py'
response = requests.get(ENGINE_URL, timeout=90)
response.raise_for_status()
ENGINE_PATH = Path('/content/renderer.py')
ENGINE_PATH.write_text(response.text, encoding='utf-8')

OUTPUT_DIR = Path('/content/drive/MyDrive/VisualNovelRenders')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base = re.sub(r'[^A-Za-z0-9_-]+', '_', ZIP_PATH.stem.replace('_colab_render_package','')) or 'VisualNovel'
OUTPUT_PATH = OUTPUT_DIR / (base + '_colab.mp4')
print('5/5 Bắt đầu render… (đã sửa nền và thời gian hoạt ảnh nhân vật)')
subprocess.run([sys.executable, '-u', str(ENGINE_PATH), '--zip', str(ZIP_PATH), '--output', str(OUTPUT_PATH)], check=True)

sidecars = [OUTPUT_PATH.with_suffix('.srt'), OUTPUT_PATH.with_name(OUTPUT_PATH.stem+'_audio_manifest.json'), OUTPUT_PATH.with_name(OUTPUT_PATH.stem+'_metadata.json')]
BUNDLE_PATH = OUTPUT_DIR / (OUTPUT_PATH.stem + '_all_files.zip')
with zipfile.ZipFile(BUNDLE_PATH, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in [OUTPUT_PATH] + sidecars:
        if path.exists(): bundle.write(path, arcname=path.name)
print('Hoàn tất! File MP4 và phụ trợ nằm tại:', OUTPUT_DIR)
print('Gói tải tất cả:', BUNDLE_PATH)

## Tải kết quả về điện thoại (tùy chọn)

Ô dưới đây tải một ZIP gồm MP4, SRT, Audio Manifest và metadata. Nếu video lớn, mở thư mục Google Drive `VisualNovelRenders` để tải sau.

In [ ]:
from google.colab import files
files.download(str(BUNDLE_PATH))